In [2]:
import pandas as pd

In [3]:
file_path = "../data/01_raw/GUIDE_Train.csv"
df_chunks = pd.read_csv(file_path,nrows=100000)

In [4]:
df_chunks.head()

,Id,OrgId,IncidentId,AlertId,Timestamp,DetectorId,AlertTitle,Category,MitreTechniques,IncidentGrade,...,ResourceType,Roles,OSFamily,OSVersion,AntispamDirection,SuspicionLevel,LastVerdict,CountryCode,State,City
0,180388628218,0,612,123247,2024-06-04T06:05:15.000Z,7,6,InitialAccess,NaN,TruePositive,...,NaN,NaN,5,66,NaN,NaN,NaN,31,6,3
1,455266534868,88,326,210035,2024-06-14T03:01:25.000Z,58,43,Exfiltration,NaN,FalsePositive,...,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630
2,1056561957389,809,58352,712507,2024-06-13T04:52:55.000Z,423,298,InitialAccess,T1189,FalsePositive,...,NaN,NaN,5,66,NaN,Suspicious,Suspicious,242,1445,10630
3,1279900258736,92,32992,774301,2024-06-10T16:39:36.000Z,2,2,CommandAndControl,NaN,BenignPositive,...,NaN,NaN,5,66,NaN,Suspicious,Suspicious,242,1445,10630
4,214748368522,148,4359,188041,2024-06-15T01:08:07.000Z,9,74,Execution,NaN,TruePositive,...,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630


In [5]:
unique_incidents = df_chunks['IncidentId'].nunique()
label_distribution = df_chunks.drop_duplicates(subset=['IncidentId'])['IncidentGrade'].value_counts(dropna=False)

print(f"Total unique incidents: {unique_incidents}")
print(f"\nLabel Distribution for Incidents:\n{label_distribution}")

Total unique incidents: 50451

Label Distribution for Incidents:
IncidentGrade
BenignPositive    26823
FalsePositive     12319
TruePositive      10836
NaN                 473
Name: count, dtype: int64


In [6]:
stats = []
for col in df_chunks.columns:
    missing_pct = df_chunks[col].isnull().mean() * 100
    unique_count = df_chunks[col].nunique()
    sample_val = df_chunks[col].dropna().iloc[0] if not df_chunks[col].dropna().empty else "ALL NULL"
    
    stats.append({
        'Feature': col,
        'Missing %': round(missing_pct, 2),
        'Unique Values': unique_count,
        'Example': sample_val
    })

stats_df = pd.DataFrame(stats).sort_values(by='Unique Values', ascending=True)
pd.set_option('display.max_rows', 50) 
print(stats_df.to_string(index=False))

           Feature  Missing %  Unique Values                  Example
      EvidenceRole       0.00              2                  Related
    SuspicionLevel      84.88              2               Suspicious
     ActionGrouped      99.41              3           ContainAccount
     IncidentGrade       0.52              3             TruePositive
 AntispamDirection      98.22              3                  Inbound
       LastVerdict      76.57              3               Suspicious
          OSFamily       0.00              4                        5
             Roles      97.73              8               Contextual
    ActionGranular      99.41             13 account password changed
         OSVersion       0.00             14                       66
      ResourceType      99.91             16          Virtual Machine
          Category       0.00             18            InitialAccess
OAuthApplicationId       0.00             18                      881
        EntityType  

In [7]:
incident_counts = df_chunks['IncidentId'].value_counts()
multi_evidence_id = incident_counts.index[0]

print(f'Inspecting IncidentId {multi_evidence_id} with {incident_counts.iloc[0]} evidence rows')

sample_cols = [
    'IncidentId', 'AlertId', 'Timestamp', 'EntityType', 
    'EvidenceRole', 'DeviceId', 'IpAddress', 'AccountName'
]

print(df_chunks[df_chunks['IncidentId'] == multi_evidence_id][sample_cols].to_string())

Inspecting IncidentId 0 with 309 evidence rows
       IncidentId  AlertId                 Timestamp         EntityType EvidenceRole  DeviceId  IpAddress  AccountName
374             0   164953  2024-06-11T08:12:42.000Z  CloudLogonRequest      Related     98799     360606       453297
1390            0   313838  2024-06-13T02:37:04.000Z                 Ip      Related     98799      18150       453297
1631            0   183418  2024-06-14T04:34:06.000Z                 Ip      Related     98799         27       453297
1634            0   332271  2024-06-13T04:36:53.000Z                 Ip      Related     98799       1178       453297
1869            0   180701  2024-06-14T09:08:43.000Z  CloudLogonRequest      Related     98799     360606       453297
2279            0   311485  2024-06-15T04:25:07.000Z  CloudLogonRequest      Related     98799     360606       453297
2559            0   183000  2024-06-11T06:40:17.000Z                 Ip      Related     98799      97863       453297
2

In [8]:
print(df_chunks.columns)

Index(['Id', 'OrgId', 'IncidentId', 'AlertId', 'Timestamp', 'DetectorId',
       'AlertTitle', 'Category', 'MitreTechniques', 'IncidentGrade',
       'ActionGrouped', 'ActionGranular', 'EntityType', 'EvidenceRole',
       'DeviceId', 'Sha256', 'IpAddress', 'Url', 'AccountSid', 'AccountUpn',
       'AccountObjectId', 'AccountName', 'DeviceName', 'NetworkMessageId',
       'EmailClusterId', 'RegistryKey', 'RegistryValueName',
       'RegistryValueData', 'ApplicationId', 'ApplicationName',
       'OAuthApplicationId', 'ThreatFamily', 'FileName', 'FolderPath',
       'ResourceIdName', 'ResourceType', 'Roles', 'OSFamily', 'OSVersion',
       'AntispamDirection', 'SuspicionLevel', 'LastVerdict', 'CountryCode',
       'State', 'City'],
      dtype='str')


In [9]:
df_chunks = df_chunks.drop(columns=["Id","ResourceType", "ActionGrouped", "ActionGranular", "ThreatFamily", "EmailClusterId", "AntispamDirection", "Roles", "SuspicionLevel", "LastVerdict", "MitreTechniques"])
# The 30 features to count
features_to_count = [
    'AlertId', 'DetectorId', 'AlertTitle', 'Category', 'EntityType', 'EvidenceRole', 
    'DeviceId', 'Sha256', 'IpAddress', 'Url', 'AccountSid', 'AccountUpn', 'AccountObjectId', 
    'AccountName', 'DeviceName', 'NetworkMessageId', 'RegistryKey', 'RegistryValueName', 
    'RegistryValueData', 'ApplicationId', 'ApplicationName', 'OAuthApplicationId', 
    'FileName', 'FolderPath', 'ResourceIdName', 'OSFamily', 'OSVersion', 
    'CountryCode', 'State', 'City'
]

aggregation_rules = {
    'OrgId': 'first',
    'IncidentGrade': 'first',
    'Timestamp': ['min', 'max']
}
for feature in features_to_count:
    aggregation_rules[feature] = 'nunique'

incident_features = df_chunks.groupby('IncidentId').agg(aggregation_rules)

In [10]:
incident_features.shape

(50451, 34)

In [11]:
unique_categories = df_chunks['Category'].dropna().unique().tolist()
print(unique_categories)

['InitialAccess', 'Exfiltration', 'CommandAndControl', 'Execution', 'SuspiciousActivity', 'Impact', 'Collection', 'CredentialAccess', 'Persistence', 'Discovery', 'Malware', 'DefenseEvasion', 'Exploit', 'PrivilegeEscalation', 'LateralMovement', 'Ransomware', 'UnwantedSoftware', 'CredentialStealing']
